# 01 — Data Exploration & Visualisation

This notebook provides an interactive walkthrough of the Sen1Floods11 dataset:
- Verifying dataset structure and file counts
- Visualising raw SAR chips and ground-truth labels
- Checking class distribution and pixel statistics
- Previewing preprocessing effects (dB conversion, normalisation)
- Previewing augmentation outputs

**Run this notebook from the project root directory.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import rasterio
import torch

from src.dataset import (
    Sen1Floods11Dataset,
    get_train_transforms,
    get_val_transforms,
    preprocess_sar,
    load_tif
)
from src.utils import set_seed

set_seed(42)

# >>>  UPDATE THIS PATH  <<<
DATA_ROOT = '../data/sen1floods11'
print('Data root:', DATA_ROOT)

## 1. Dataset split sizes

In [ ]:
for split in ['train', 'val', 'test']:
    ds = Sen1Floods11Dataset(DATA_ROOT, split=split,
                              transforms=get_val_transforms())
    print(f'{split:6s}: {len(ds):4d} chips')

## 2. Visualise raw + preprocessed chip

In [ ]:
ds = Sen1Floods11Dataset(DATA_ROOT, split='train',
                          transforms=get_val_transforms())

# Load a raw chip (before preprocessing)
raw_chip = load_tif(ds.image_paths[0])   # (H, W, 2)
raw_mask = load_tif(ds.mask_paths[0])    # (H, W, 1)

# Preprocess
proc_chip = preprocess_sar(raw_chip)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Raw vs Preprocessed SAR Chip', fontsize=14, fontweight='bold')

titles_raw  = ['Raw VV (linear)', 'Raw VH (linear)', 'Label mask']
titles_proc = ['Preprocessed VV (normalised)', 'Preprocessed VH (normalised)', 'Label mask']

for i, (t, img) in enumerate(zip(titles_raw, [raw_chip[...,0], raw_chip[...,1], raw_mask[...,0]])):
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(t); axes[0, i].axis('off')

for i, (t, img) in enumerate(zip(titles_proc, [proc_chip[...,0], proc_chip[...,1], raw_mask[...,0]])):
    axes[1, i].imshow(img, cmap='gray')
    axes[1, i].set_title(t); axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('../outputs/data_exploration_chip.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Class distribution

In [ ]:
flooded_fracs = []
for img_path, mask_path in zip(ds.image_paths[:100], ds.mask_paths[:100]):
    mask = load_tif(mask_path)[..., 0]
    valid = mask != -1
    frac  = (mask[valid] == 1).mean() if valid.sum() > 0 else 0.0
    flooded_fracs.append(frac)

flooded_fracs = np.array(flooded_fracs)
print(f'Mean flooded fraction: {flooded_fracs.mean():.3f}')
print(f'Median flooded fraction: {np.median(flooded_fracs):.3f}')

plt.figure(figsize=(8, 4))
plt.hist(flooded_fracs, bins=30, color='steelblue', edgecolor='white')
plt.xlabel('Fraction of flooded pixels per chip')
plt.ylabel('Count')
plt.title('Class distribution across training chips (first 100)')
plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Augmentation preview

In [ ]:
aug_ds = Sen1Floods11Dataset(DATA_ROOT, split='train',
                               transforms=get_train_transforms())

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Augmentation samples (same chip, different augmentations)', fontsize=12)

for col in range(4):
    img, mask = aug_ds[0]   # repeatedly load chip 0 to see random augmentations
    axes[0, col].imshow(img[0].numpy(), cmap='gray')
    axes[0, col].set_title(f'VV — aug {col+1}'); axes[0, col].axis('off')
    axes[1, col].imshow(mask[0].numpy(), cmap='Blues', vmin=0, vmax=1)
    axes[1, col].set_title(f'Mask — aug {col+1}'); axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('../outputs/augmentation_preview.png', dpi=150, bbox_inches='tight')
plt.show()